# EARC — Module 1: Retrieval Layer (v4)
**Evidence-Aware Context Compression for Token-Efficient RAG**

Stages 1–3 per the 13-stage High Level Design:
- **Stage 1** — Query Analysis & Classification
- **Stage 2** — Adaptive Hybrid Retrieval (BM25 + FAISS + RRF)
- **Stage 3** — Sentence Segmentation & Metadata Attachment

**Changes in v4 over v3:**
1. **Stronger classification** — detects multiple WH-words, coordinating conjunctions, comparison/contrast signals, and compound queries beyond just entity count and verb count
2. **Negation flag** — `has_negation` added to `query_info`; Module 2/3 can use it for scoring/filtering
3. **`None` embeddings** — zero-array placeholder replaced with `None`; Module 2 fills in real embeddings
4. **`nlp.pipe()` batch processing** — sentence-level spaCy calls batched for performance
5. **BM25 + FAISS scores stored** — individual retrieval scores kept alongside RRF score for Module 2
6. **Stable sentence IDs** — `sentence_id = dataset:doc_id:chunk_id:position` on every SentenceObject
7. **`retrieval_score` field reserved** — Module 2 can use raw retrieval strength as a scoring signal
8. **Granular timing logs** — BM25 time, FAISS time, RRF time, segmentation time logged separately
9. **`export_for_module2()` marked as debug utility only** — production path is in-RAM handoff

## Cell 1 — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/RAG_Project'
print('Files in RAG_Project:', os.listdir(PROJECT_DIR))

In [ ]:
!pip install -q rank_bm25 spacy faiss-cpu sentence-transformers
!python -m spacy download en_core_web_sm -q
print('Dependencies ready.')

## Cell 2 — Imports

In [ ]:
import pickle
import re
import time
import logging
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import faiss
import numpy as np
import spacy
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('EARC-M1')

nlp = spacy.load('en_core_web_sm')
log.info('spaCy loaded: %s', nlp.meta['name'])

## Cell 3 — Configuration

All tunable parameters in one place. These will move to `retrieval/retrieval_config.py` in the GitHub repo so every module imports from the same source of truth.

In [ ]:
DRIVE_BASE   = Path('/content/drive/MyDrive/RAG_Project')
FAISS_PATH   = DRIVE_BASE / 'faiss.index'
BM25_PATH    = DRIVE_BASE / 'bm25.pkl'
CHUNKS_DIR   = DRIVE_BASE / 'chunks'
METADATA_DIR = DRIVE_BASE / 'metadata'

EMBED_MODEL  = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIM    = 384
RRF_K        = 60  # standard RRF smoothing constant

# Retrieval depth per query type.
# Controls BM25/FAISS candidate pool sizes and final RRF top-N.
# Token budget for the LLM prompt is handled downstream in Module 3.
K_BY_TYPE = {
    'factoid'    : {'bm25': 15, 'faiss': 15, 'final': 8},
    'descriptive': {'bm25': 20, 'faiss': 20, 'final': 12},
    'multi_hop'  : {'bm25': 30, 'faiss': 30, 'final': 20},
}

# Sentence length bounds for Stage 3 segmentation.
MIN_SENT_TOKENS = 5    # drops fragment labels and bullet artefacts
MAX_SENT_TOKENS = 150  # drops garbled OCR rows and HTML table cells

# spaCy batch size for nlp.pipe() in Stage 3.
SPACY_BATCH_SIZE = 32

# ── Closed linguistic sets — do not change unless English grammar changes ──────

# Named entity types that count as substantive query subjects.
# Incidental GPE/LOC/DATE/TIME/CARDINAL alone do NOT trigger multi_hop.
_SUBSTANTIVE_ENT_TYPES = {
    'PERSON', 'ORG', 'WORK_OF_ART', 'EVENT', 'PRODUCT', 'LAW', 'NORP', 'FAC'
}

# POS tags for BM25 content keywords.
_CONTENT_POS = {'NOUN', 'PROPN', 'VERB', 'ADJ', 'NUM'}

# Wh-words by query category.
_FACTOID_WH     = {'who', 'what', 'when', 'where', 'which', 'whom'}
_DESCRIPTIVE_WH = {'how', 'why'}
_ALL_WH         = _FACTOID_WH | _DESCRIPTIVE_WH

# Dependency labels marking auxiliary verbs — excluded from keywords.
_AUX_DEPS = {'aux', 'auxpass'}

# Clause-level dependency labels for finite verb detection.
_CLAUSE_DEPS = {'ROOT', 'relcl', 'ccomp', 'advcl', 'acl'}

# Comparison/contrast lexical signals — presence suggests multi-entity reasoning.
# These are surface forms, checked after lowercasing.
_COMPARISON_TOKENS = {
    'compare', 'comparison', 'versus', 'vs', 'difference',
    'differences', 'contrast', 'similar', 'both', 'neither',
    'between', 'respectively',
}

# Negation tokens — used to set has_negation flag (NOT stripped from query).
_NEGATION_TOKENS = {'not', 'never', 'no', 'except', 'without', 'excluding',
                    'neither', 'nor', 'non', 'non-member', 'outside'}

# Characters that mark chunk boundary fragments or list artefacts.
_FRAGMENT_START_CHARS = set('abcdefghijklmnopqrstuvwxyz*•–—|·')

# Leading determiners to strip from entity spans.
_DETERMINERS = {'the', 'a', 'an'}

print('Configuration loaded.')

## Cell 4 — SentenceObject Dataclass

Shared data model for the full EARC pipeline. Module 1 populates structural fields. Module 2 fills in scores and real embeddings. Module 3 sets `force_include`.

In [ ]:
@dataclass
class SentenceObject:
    """
    One sentence flowing through the EARC pipeline.

    Module 1 sets: all structural/identity fields + contains_query_entity.
    Module 2 sets: embedding (real), all score fields.
    Module 3 sets: force_include.

    sentence_id is a stable string identifier:
        '{dataset}:{doc_id}:{chunk_id}:{position}'
    Used for deduplication, debugging, and evaluation logging.

    embedding is None until Module 2 fills it in.
    Storing None instead of np.zeros(384) saves ~1.5 KB per sentence
    across 100+ candidate sentences per query.

    retrieval_score is the normalised RRF score from Module 1.
    Reserved for Module 2 to use as an additional ranking signal.
    """
    # ── Identity / structural (Module 1) ──────────────────────────────────────
    sentence_id           : str            # stable: dataset:doc_id:chunk_id:position
    text                  : str
    doc_id                : str
    dataset               : str
    title                 : str
    position              : int            # sentence index within its chunk
    retrieval_rank        : int            # 1-based RRF rank of parent chunk
    chunk_id              : int            # index into all_chunks / FAISS
    year                  : Optional[int]  # document year for temporal scoring
    bm25_score            : float          # raw BM25 score of parent chunk
    faiss_score           : float          # raw FAISS cosine score of parent chunk
    retrieval_score       : float          # normalised RRF score (reserved for Module 2)
    contains_query_entity : bool           # entity OR keyword overlap with query
    token_count           : int            # whitespace-split word count

    # ── Embedding (Module 2 fills this) ───────────────────────────────────────
    embedding             : Optional[np.ndarray] = None  # 384-dim; None until Module 2

    # ── Scores (Module 2 fills these) ─────────────────────────────────────────
    semantic_score        : float = 0.0
    evidence_score        : float = 0.0
    evidentiality_score   : float = 0.0
    claim_density_score   : float = 0.0
    temporal_score        : float = 0.0
    final_score           : float = 0.0

    # ── Selection flag (Module 3 sets this) ───────────────────────────────────
    force_include         : bool = False

    def __repr__(self):
        return (
            f'SentenceObject(id={self.sentence_id!r}, '
            f'rank={self.retrieval_rank}, score={self.final_score:.3f}, '
            f'entity={self.contains_query_entity}, '
            f'text={self.text[:60]!r})'
        )

print('SentenceObject defined.')

## Cell 5 — Load Corpus Artifacts

In [ ]:
def load_corpus_artifacts(
    faiss_path, bm25_path, chunks_dir, metadata_dir, embed_model_name
):
    t0 = time.time()

    log.info('Loading FAISS index ...')
    faiss_index = faiss.read_index(str(faiss_path))
    log.info('  FAISS: %d vectors, dim=%d', faiss_index.ntotal, faiss_index.d)
    assert faiss_index.d == EMBED_DIM, (
        f'FAISS dim mismatch: index has {faiss_index.d}, config expects {EMBED_DIM}'
    )

    log.info('Loading BM25 index ...')
    with open(bm25_path, 'rb') as f:
        bm25_index = pickle.load(f)
    log.info('  BM25: %d documents', len(bm25_index.idf))

    log.info('Loading chunk shards ...')
    all_chunks = []
    for shard_path in sorted(chunks_dir.glob('chunks_*.pkl')):
        with open(shard_path, 'rb') as f:
            all_chunks.extend(pickle.load(f))
        log.info('  %s: running total %d', shard_path.name, len(all_chunks))
    assert len(all_chunks) == faiss_index.ntotal, (
        f'Chunk/FAISS mismatch: {len(all_chunks)} chunks vs {faiss_index.ntotal} vectors'
    )

    log.info('Loading metadata shards ...')
    all_metadata = []
    for meta_path in sorted(metadata_dir.glob('metadata_*.pkl')):
        with open(meta_path, 'rb') as f:
            all_metadata.extend(pickle.load(f))
        log.info('  %s: running total %d', meta_path.name, len(all_metadata))
    assert len(all_metadata) == len(all_chunks), (
        f'Metadata/chunk mismatch: {len(all_metadata)} vs {len(all_chunks)}'
    )

    log.info('Loading embedding model: %s ...', embed_model_name)
    model = SentenceTransformer(embed_model_name)
    log.info('  dim=%d', model.get_sentence_embedding_dimension())

    log.info('All artifacts loaded in %.1fs', time.time() - t0)
    return faiss_index, bm25_index, all_chunks, all_metadata, model


faiss_index, bm25_index, all_chunks, all_metadata, embed_model = load_corpus_artifacts(
    FAISS_PATH, BM25_PATH, CHUNKS_DIR, METADATA_DIR, EMBED_MODEL
)
print(f'Corpus ready: {len(all_chunks):,} chunks.')

## Cell 6 — Stage 1: QueryAnalyzer

Classification logic (priority order):
1. **`multi_hop`** — 2+ substantive entities, OR 2+ finite verbs with a WH-word, OR multiple WH-words, OR explicit comparison/contrast tokens, OR compound conjunction between two question clauses
2. **`factoid`** — single factoid WH-word, single finite verb, no multi_hop signals
3. **`descriptive`** — how/why, no WH-word, or long query

Negation detection is **separate from classification**: `has_negation` is always set in `query_info` regardless of query type, so Module 2/3 can use it without re-inspecting the query.

In [ ]:
class QueryAnalyzer:
    """
    Stage 1 — Query Analysis & Classification.

    Returns a dict with keys:
        query        : str        — original query string (propagated to all modules)
        query_type   : str        — 'factoid' | 'multi_hop' | 'descriptive'
        keywords     : List[str]  — lemmatised content words for BM25
        entities     : List[str]  — named entity strings for Stage 9 entity guard
        has_negation : bool       — True if query contains a negation signal
    """

    def __init__(self):
        self._nlp = nlp

    # ── Public entry point ────────────────────────────────────────────────────

    def analyze(self, query: str) -> Dict:
        doc          = self._nlp(query)
        has_negation = self._detect_negation(doc)
        query_type   = self._classify(doc)
        keywords     = self._extract_keywords(doc)
        entities     = self._extract_entities(doc)

        log.info(
            'QueryAnalyzer: type=%-12s negation=%-5s | keywords=%s | entities=%s',
            query_type, has_negation, keywords, entities
        )
        return {
            'query'       : query,
            'query_type'  : query_type,
            'keywords'    : keywords,
            'entities'    : entities,
            'has_negation': has_negation,
        }

    # ── Negation detection ────────────────────────────────────────────────────

    def _detect_negation(self, doc) -> bool:
        """
        Return True if the query contains a negation or exclusion signal.

        Checks both:
        - spaCy dep_ == 'neg' (syntactic negation: 'not', 'never', 'no')
        - Lexical set _NEGATION_TOKENS (catches 'except', 'without',
          'excluding', 'non-member', 'outside' which spaCy may not
          parse as negation)

        has_negation is carried in query_info for Module 2/3 to use.
        It does NOT affect BM25 keywords (BM25 cannot handle negation).
        It does NOT affect the query string sent to FAISS (full query always used).
        """
        for token in doc:
            if token.dep_ == 'neg':
                return True
            if token.text.lower() in _NEGATION_TOKENS:
                return True
        return False

    # ── Classification ────────────────────────────────────────────────────────

    def _classify(self, doc) -> str:
        """
        Classify query type using spaCy parse output.

        multi_hop signals (any one is sufficient):
          A. 2+ substantive named entities (cross-doc evidence needed)
          B. 2+ finite verbs AND a factoid WH-word (compound question:
             'Which film won Best Picture and who directed it?')
          C. 2+ WH-words of any kind
             ('When and where was Tesla born?' has 'when' + 'where')
          D. Explicit comparison/contrast token present
             ('Compare Newton and Einstein', 'difference between X and Y')
          E. Coordinating conjunction between two named entities OR
             two question clauses (heuristic: CC dep + 2 content branches)

        factoid: single factoid WH-word, no multi_hop signals.
        descriptive: how/why, no WH-word, or neither of the above.
        """
        token_texts_lower = [t.text.lower() for t in doc]
        token_set         = set(token_texts_lower)

        # Substantive named entities only
        subst_entities = [
            ent for ent in doc.ents
            if ent.label_ in _SUBSTANTIVE_ENT_TYPES
        ]

        # Finite clause verbs (ROOT / relcl / ccomp / advcl / acl)
        finite_verbs = [
            t for t in doc
            if t.pos_ == 'VERB' and t.dep_ in _CLAUSE_DEPS
        ]

        # Count WH-words
        wh_words_found = [w for w in token_texts_lower if w in _ALL_WH]

        has_factoid_wh     = bool(token_set & _FACTOID_WH)
        has_descriptive_wh = bool(token_set & _DESCRIPTIVE_WH)
        has_comparison     = bool(token_set & _COMPARISON_TOKENS)

        # ── multi_hop signals ──────────────────────────────────────────────────

        # A. Two or more substantive named entities
        if len(subst_entities) >= 2:
            return 'multi_hop'

        # B. Compound question: 2+ finite verbs with a factoid WH-word
        if len(finite_verbs) >= 2 and has_factoid_wh:
            return 'multi_hop'

        # C. Multiple WH-words: 'When and where', 'Who and what'
        if len(set(wh_words_found)) >= 2:
            return 'multi_hop'

        # D. Comparison/contrast token — implies reasoning across entities
        if has_comparison:
            return 'multi_hop'

        # E. Coordinating conjunction joining two clause branches
        # Heuristic: 'and'/'or' appears AND there are 2+ finite verbs
        # (catches 'Who invented C++ and when did it become popular?')
        cc_tokens = [t for t in doc if t.dep_ == 'cc']
        if cc_tokens and len(finite_verbs) >= 2:
            return 'multi_hop'

        # ── factoid ────────────────────────────────────────────────────────────
        if has_factoid_wh:
            return 'factoid'

        # ── descriptive ────────────────────────────────────────────────────────
        return 'descriptive'

    # ── Keyword extraction ────────────────────────────────────────────────────

    def _extract_keywords(self, doc) -> List[str]:
        """
        BM25 query tokens: lemmatised content words.

        Two refinements:
        1. Named entity spans preserved as single lowercased units
           ('Marie Curie' → 'marie curie', not 'marie' + 'curie').
        2. Auxiliary verbs (dep_=aux/auxpass) excluded even though
           spaCy tags them as VERB.

        Negation tokens (ADV/PART) are excluded from BM25 keywords
        since BM25 cannot handle negation — but they are NOT stripped
        from the query string; has_negation flag carries that information.
        """
        keywords = []
        ent_token_indices = set()

        for ent in doc.ents:
            ent_token_indices.update(range(ent.start, ent.end))
            span_text = ent.text.lower().strip()
            if len(span_text) > 1:
                keywords.append(span_text)

        for token in doc:
            if token.i in ent_token_indices:
                continue
            if token.pos_ not in _CONTENT_POS:
                continue
            if token.dep_ in _AUX_DEPS:
                continue
            if token.is_punct or token.is_space:
                continue
            if len(token.text) <= 1:
                continue
            keywords.append(token.lemma_.lower())

        # Deduplicate, preserve order
        seen, deduped = set(), []
        for kw in keywords:
            if kw not in seen:
                seen.add(kw)
                deduped.append(kw)
        return deduped

    # ── Entity extraction ─────────────────────────────────────────────────────

    def _extract_entities(self, doc) -> List[str]:
        """
        All named entities for Stage 9 entity guard.
        Leading determiners stripped; deduplicated (case-insensitive).
        All types included (dates/numbers matter for entity guard).
        """
        seen, result = set(), []
        for ent in doc.ents:
            words = ent.text.strip().split()
            if words and words[0].lower() in _DETERMINERS:
                words = words[1:]
            text = ' '.join(words).strip()
            if text and text.lower() not in seen:
                seen.add(text.lower())
                result.append(text)
        return result


# ── Classification test ───────────────────────────────────────────────────────
qa = QueryAnalyzer()
test_queries = [
    # Core cases
    ('Who invented the telephone?',                                              'factoid'),
    ('What is the capital of France?',                                           'factoid'),
    ('How does photosynthesis work?',                                            'descriptive'),
    ('What did Marie Curie and Albert Einstein both contribute to physics?',     'multi_hop'),
    ('Which film won the Academy Award for Best Picture in 2020 and who directed it?', 'multi_hop'),
    ('What countries are not members of NATO?',                                  'factoid'),
    # v4 new cases
    ('In which year did the Berlin Wall fall?',                                  'factoid'),
    ('What did Einstein publish in Germany in 1905?',                            'factoid'),
    ('Why did the Roman Empire collapse?',                                       'descriptive'),
    ('When and where was Nikola Tesla born?',                                    'multi_hop'),
    ('Who invented C++ and when did it become popular?',                         'multi_hop'),
    ('Compare Newton and Einstein on gravity',                                   'multi_hop'),
    ('What is the difference between TCP and UDP?',                              'multi_hop'),
]

print(f'{"Expected":<14} {"Got":<14} {"Match":<6}  Query')
print('-' * 90)
all_pass = True
for q, expected in test_queries:
    r     = qa.analyze(q)
    got   = r['query_type']
    match = '✓' if got == expected else '✗ FAIL'
    if got != expected:
        all_pass = False
    neg_flag = f'  [neg={r["has_negation"]}]' if r['has_negation'] else ''
    print(f'  {expected:<14} {got:<14} {match:<6}  {q}{neg_flag}')
print()
print('All classification tests passed.' if all_pass else 'WARNING: some tests failed — review classifier.')

## Cell 7 — Stage 2: HybridRetriever (BM25 + FAISS + RRF)

Individual BM25 and FAISS scores are now stored per chunk in addition to the RRF score. This lets Module 2's SentenceScorer use raw retrieval strength as an additional signal without re-running retrieval.

Granular timing is logged separately for BM25, FAISS, and RRF merge steps.

In [ ]:
class HybridRetriever:
    """
    Stage 2 — Adaptive Hybrid Retrieval.

    BM25 (sparse, keyword) + FAISS (dense, semantic) merged by RRF.
    RRF score: rrf(c) = Σ_r [ 1 / (k + rank_r(c)) ]

    Individual bm25_score and faiss_score are stored per chunk
    alongside rrf_score for Module 2 to use as retrieval signals.
    """

    def __init__(self, faiss_index, bm25_index, all_chunks, all_metadata, model):
        self._faiss    = faiss_index
        self._bm25     = bm25_index
        self._chunks   = all_chunks
        self._metadata = all_metadata
        self._model    = model

    # ── BM25 ──────────────────────────────────────────────────────────────────

    def _retrieve_bm25(
        self, keywords: List[str], top_k: int
    ) -> Tuple[List[Tuple[int, float]], float]:
        """
        Query BM25 with content keywords.
        Multi-word entity keywords are split into tokens (BM25 is token-level).
        Returns (chunk_idx, score) list and wall-clock time.
        """
        t0 = time.time()
        if not keywords:
            return [], time.time() - t0
        bm25_tokens = []
        for kw in keywords:
            bm25_tokens.extend(kw.split())
        scores      = self._bm25.get_scores(bm25_tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        results     = [(int(i), float(scores[i])) for i in top_indices if scores[i] > 0]
        return results, time.time() - t0

    # ── FAISS ─────────────────────────────────────────────────────────────────

    def _retrieve_faiss(
        self, query: str, top_k: int
    ) -> Tuple[List[Tuple[int, float]], float]:
        """
        Embed full query and search FAISS IndexFlatIP.
        normalize_embeddings=True → inner product == cosine similarity.
        Returns (chunk_idx, cosine_score) list and wall-clock time.
        """
        t0    = time.time()
        q_vec = self._model.encode(
            [query], normalize_embeddings=True, show_progress_bar=False
        ).astype('float32')
        scores, indices = self._faiss.search(q_vec, top_k)
        results = [
            (int(idx), float(score))
            for idx, score in zip(indices[0], scores[0])
            if idx >= 0
        ]
        return results, time.time() - t0

    # ── RRF merge ─────────────────────────────────────────────────────────────

    def _rrf_merge(
        self,
        bm25_results  : List[Tuple[int, float]],
        faiss_results : List[Tuple[int, float]],
        top_k_final   : int,
    ) -> Tuple[List[Tuple[int, float]], float]:
        """
        Reciprocal Rank Fusion.
        Chunks in both lists accumulate higher RRF scores.
        Returns merged list and wall-clock time.
        """
        t0     = time.time()
        scores : Dict[int, float] = {}
        for rank0, (idx, _) in enumerate(bm25_results):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (RRF_K + rank0 + 1)
        for rank0, (idx, _) in enumerate(faiss_results):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (RRF_K + rank0 + 1)
        merged = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k_final]
        return merged, time.time() - t0

    # ── Main entry point ──────────────────────────────────────────────────────

    def fused_retrieve(self, query: str, query_info: Dict) -> List[Dict]:
        """
        Run BM25 + FAISS + RRF. Returns chunk dicts containing:
            chunk_idx, chunk_text, rrf_rank, rrf_score,
            bm25_score, faiss_score,
            doc_id, dataset, title, year
        """
        qt    = query_info['query_type']
        kw    = query_info['keywords']
        k_cfg = K_BY_TYPE.get(qt)
        if k_cfg is None:
            log.warning('Unknown query type %r — falling back to factoid depths', qt)
            k_cfg = K_BY_TYPE['factoid']

        bm25_r,  t_bm25  = self._retrieve_bm25(kw, k_cfg['bm25'])
        faiss_r, t_faiss = self._retrieve_faiss(query, k_cfg['faiss'])
        merged,  t_rrf   = self._rrf_merge(bm25_r, faiss_r, k_cfg['final'])

        log.info(
            'Retrieval [%s]: BM25=%d(%.3fs) FAISS=%d(%.3fs) RRF=%d(%.3fs)',
            qt,
            len(bm25_r),  t_bm25,
            len(faiss_r), t_faiss,
            len(merged),  t_rrf,
        )

        # Build lookup dicts for individual scores
        bm25_score_map  = {idx: score for idx, score in bm25_r}
        faiss_score_map = {idx: score for idx, score in faiss_r}

        results = []
        for rank0, (chunk_idx, rrf_score) in enumerate(merged):
            meta = self._metadata[chunk_idx]
            results.append({
                'chunk_idx'   : chunk_idx,
                'chunk_text'  : self._chunks[chunk_idx],
                'rrf_score'   : rrf_score,
                'rrf_rank'    : rank0 + 1,
                'bm25_score'  : bm25_score_map.get(chunk_idx, 0.0),
                'faiss_score' : faiss_score_map.get(chunk_idx, 0.0),
                'doc_id'      : meta.get('doc_id', ''),
                'dataset'     : meta.get('dataset', ''),
                'title'       : meta.get('title', ''),
                'year'        : meta.get('year', None),
            })
        return results


retriever = HybridRetriever(faiss_index, bm25_index, all_chunks, all_metadata, embed_model)
print('HybridRetriever ready.')

## Cell 8 — Stage 3: Sentence Segmentation → SentenceObject

Key changes in v4:
- **`nlp.pipe()` batch processing** — all chunk texts processed in one batched spaCy call instead of one-by-one, significantly faster on large retrieved sets
- **`sentence_id`** assigned as `dataset:doc_id:chunk_id:position`
- **`bm25_score`, `faiss_score`, `retrieval_score`** propagated from chunk dict into SentenceObject
- **`embedding=None`** instead of `np.zeros(384)`

In [ ]:
_WHITESPACE_RE = re.compile(r'\s+')


def _clean(text: str) -> str:
    return _WHITESPACE_RE.sub(' ', text).strip()


def _is_fragment(text: str) -> bool:
    """
    True if text looks like a chunk boundary fragment or list artefact.
    Catches: lowercase starts (mid-sentence cuts) and bullet/dash artefacts.
    """
    if not text:
        return True
    return text[0] in _FRAGMENT_START_CHARS


def _approx_tokens(text: str) -> int:
    """Whitespace-split word count as token approximation."""
    return len(text.split())


def _has_entity_or_keyword(
    sentence : str,
    entities : List[str],
    keywords : List[str],
    sent_doc,            # pre-computed spaCy doc for this sentence
) -> bool:
    """
    True if sentence contains any query entity (substring) or keyword (lemma).

    Level 1: entity substring match (case-insensitive).
    Level 2: keyword lemma match against sentence token lemmas.
             Handles morphological variants: 'telephones'→'telephone',
             'invented'→'invent', 'members'→'member'.

    sent_doc is passed in (pre-computed by nlp.pipe batch) to avoid
    redundant spaCy calls.
    """
    s_lower = sentence.lower()
    if any(ent.lower() in s_lower for ent in entities if ent):
        return True
    if keywords and sent_doc is not None:
        sent_lemmas = {
            t.lemma_.lower() for t in sent_doc
            if not t.is_punct and not t.is_space
        }
        if any(kw in sent_lemmas for kw in keywords):
            return True
    return False


def segment_to_sentences(
    retrieved_chunks : List[Dict],
    query_entities   : List[str],
    query_keywords   : List[str],
    min_tokens       : int = MIN_SENT_TOKENS,
    max_tokens       : int = MAX_SENT_TOKENS,
) -> List[SentenceObject]:
    """
    Stage 3 — Sentence Segmentation & Metadata Attachment.

    Uses nlp.pipe() to process all chunk texts in a single batched spaCy
    call (senter + tagger + lemmatizer only — no NER, no parser).

    For each sentence:
      1. Fragment filter
      2. Length filter
      3. Entity/keyword overlap flag (using pre-computed sentence lemmas)
      4. SentenceObject creation with all Module 1 fields set

    Returns List[SentenceObject] with embedding=None (Module 2 fills this).
    """
    if not retrieved_chunks:
        log.warning('segment_to_sentences: no chunks received.')
        return []

    t0 = time.time()

    # Batch all chunk texts through spaCy in one pass.
    # Keep: parser (sentence boundaries in en_core_web_sm come from the parser,
    #         not a standalone senter), tagger, attribute_ruler, lemmatizer.
    # Disable: ner only — not needed at segmentation time.
    chunk_texts = [chunk['chunk_text'] for chunk in retrieved_chunks]
    spacy_docs  = list(nlp.pipe(
        chunk_texts,
        batch_size = SPACY_BATCH_SIZE,
        disable    = ['ner'],   # keep parser: en_core_web_sm uses it for .sents
    ))

    results     = []
    n_fragments = 0
    n_too_short = 0
    n_too_long  = 0

    for chunk, spacy_doc in zip(retrieved_chunks, spacy_docs):
        for sent_idx, sent in enumerate(spacy_doc.sents):
            text = _clean(sent.text)
            if not text:
                continue

            if _is_fragment(text):
                n_fragments += 1
                continue

            tok_count = _approx_tokens(text)
            if tok_count < min_tokens:
                n_too_short += 1
                continue
            if tok_count > max_tokens:
                n_too_long += 1
                continue

            # Build a lightweight spaCy span doc for lemma-based keyword matching.
            # We use the already-computed spacy_doc and extract the span.
            sent_span_doc = sent.as_doc()

            has_entity = _has_entity_or_keyword(
                text, query_entities, query_keywords, sent_span_doc
            )

            # Stable sentence ID
            sentence_id = (
                f"{chunk['dataset']}:{chunk['doc_id']}"
                f":{chunk['chunk_idx']}:{sent_idx}"
            )

            results.append(SentenceObject(
                sentence_id           = sentence_id,
                text                  = text,
                doc_id                = chunk['doc_id'],
                dataset               = chunk['dataset'],
                title                 = chunk['title'],
                position              = sent_idx,
                retrieval_rank        = chunk['rrf_rank'],
                chunk_id              = chunk['chunk_idx'],
                year                  = chunk.get('year', None),
                bm25_score            = chunk['bm25_score'],
                faiss_score           = chunk['faiss_score'],
                retrieval_score       = chunk['rrf_score'],
                embedding             = None,          # Module 2 fills this
                contains_query_entity = has_entity,
                token_count           = tok_count,
            ))

    t_seg = time.time() - t0
    log.info(
        'Segmentation: %d sentences from %d chunks in %.3fs '
        '(dropped: %d fragments, %d too short, %d too long)',
        len(results), len(retrieved_chunks), t_seg,
        n_fragments, n_too_short, n_too_long,
    )
    return results


print('segment_to_sentences() defined.')

## Cell 9 — RetrievalLayer: Full Stage 1–3 Interface

In [ ]:
class RetrievalLayer:
    """
    Module 1 top-level interface — Stages 1, 2, 3.

    Production usage (in-RAM handoff):
        sentences, query_info = layer.retrieve(query)
        # pass directly to Module 2 — no pickle, no disk I/O

    sentences  → List[SentenceObject]  (Module 2 fills embeddings + scores)
    query_info → dict {query, query_type, keywords, entities, has_negation}
                 propagated unchanged through all downstream modules
    """

    def __init__(self, faiss_index, bm25_index, all_chunks, all_metadata, model):
        self.analyzer  = QueryAnalyzer()
        self.retriever = HybridRetriever(
            faiss_index, bm25_index, all_chunks, all_metadata, model
        )

    def retrieve(self, query: str) -> Tuple[List[SentenceObject], Dict]:
        log.info('=' * 60)
        log.info('Query: %r', query)
        t0 = time.time()

        query_info = self.analyzer.analyze(query)
        chunks     = self.retriever.fused_retrieve(query, query_info)
        sentences  = segment_to_sentences(
            chunks,
            query_info['entities'],
            query_info['keywords'],
        )

        log.info(
            'Module 1 total: %.2fs → %d sentences | type=%s | negation=%s',
            time.time() - t0,
            len(sentences),
            query_info['query_type'],
            query_info['has_negation'],
        )
        log.info('=' * 60)
        return sentences, query_info


retrieval_layer = RetrievalLayer(
    faiss_index, bm25_index, all_chunks, all_metadata, embed_model
)
print('RetrievalLayer ready.')

## Cell 10 — Test: Factoid Query (telephone)

In [ ]:
query = 'Who invented the telephone?'
sentences, qi = retrieval_layer.retrieve(query)

print(f'Query        : {query}')
print(f'Type         : {qi["query_type"]}')
print(f'Keywords     : {qi["keywords"]}')
print(f'Entities     : {qi["entities"]}')
print(f'Has negation : {qi["has_negation"]}')
print(f'Sentences    : {len(sentences)}')
entity_sents = sum(1 for s in sentences if s.contains_query_entity)
print(f'With entity/keyword match: {entity_sents}')
print()
print('Top 5 sentences:')
for i, s in enumerate(sentences[:5]):
    print(f'  [{i+1}] rank={s.retrieval_rank} | entity={s.contains_query_entity} | '
          f'tokens={s.token_count} | year={s.year} | bm25={s.bm25_score:.2f} | faiss={s.faiss_score:.3f}')
    print(f'       id : {s.sentence_id}')
    print(f'       {s.text}')
    print()

## Cell 11 — Test: Multi-hop Query (Curie + Einstein)

In [ ]:
query = 'What did Marie Curie and Albert Einstein both contribute to physics?'
sentences, qi = retrieval_layer.retrieve(query)

print(f'Query        : {query}')
print(f'Type         : {qi["query_type"]}')
print(f'Keywords     : {qi["keywords"]}')
print(f'Entities     : {qi["entities"]}')
print(f'Has negation : {qi["has_negation"]}')
print(f'Sentences    : {len(sentences)}')
print()
print('Top 5 sentences:')
for i, s in enumerate(sentences[:5]):
    print(f'  [{i+1}] rank={s.retrieval_rank} | dataset={s.dataset} | entity={s.contains_query_entity}')
    print(f'       id : {s.sentence_id}')
    print(f'       {s.text}')
    print()

## Cell 12 — Test: Negation query (NATO)

In [ ]:
query = 'What countries are not members of NATO?'
sentences, qi = retrieval_layer.retrieve(query)

print(f'Query        : {query}')
print(f'Type         : {qi["query_type"]}')
print(f'Keywords     : {qi["keywords"]}')
print(f'Entities     : {qi["entities"]}')
print(f'Has negation : {qi["has_negation"]}   ← should be True')
print(f'"not" in keywords: {"not" in qi["keywords"]}   ← should be False (ADV excluded from BM25)')
print(f'Sentences    : {len(sentences)}')
print()
for i, s in enumerate(sentences[:3]):
    print(f'  [{i+1}] {s.text}')
    print()

## Cell 13 — Test: v4 edge cases (comparison, compound WH, CC-conjunction)

In [ ]:
edge_cases = [
    ('Who invented C++ and when did it become popular?', 'multi_hop'),
    ('Compare Newton and Einstein on gravity',            'multi_hop'),
    ('What is the difference between TCP and UDP?',      'multi_hop'),
    ('In which year did the Berlin Wall fall?',           'factoid'),
    ('What did Einstein publish in Germany in 1905?',    'factoid'),
    ('Why did the Roman Empire collapse?',               'descriptive'),
    ('When and where was Nikola Tesla born?',            'multi_hop'),
]

qa = QueryAnalyzer()
print(f'{"Expected":<14} {"Got":<14} {"Neg":<5}  Query')
print('-' * 80)
for q, expected in edge_cases:
    r   = qa.analyze(q)
    got = r['query_type']
    ok  = '✓' if got == expected else '✗ FAIL'
    print(f'  {expected:<14} {got:<14} {str(r["has_negation"]):<5}  {ok}  {q}')

## Cell 14 — Debug utility: Export for Module 2

**This is a debugging and development utility only.**

In the production pipeline (`pipeline.py`), Module 1's output is handed directly to Module 2 in RAM — no pickle, no disk I/O. Use this cell only when you need to inspect Module 1 output in isolation or share it with another team member for Module 2 development.

In [ ]:
OUTPUT_DIR = DRIVE_BASE / 'earc_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def export_for_module2(query: str) -> Path:
    """
    DEBUG UTILITY — not called in production pipeline.

    Save Module 1 output to disk for Module 2 to load independently.
    File: earc_outputs/m1_{slug}.pkl
    Contents: {query, query_info, sentences}
    """
    sentences, query_info = retrieval_layer.retrieve(query)
    slug     = re.sub(r'[^a-z0-9]+', '_', query.lower())[:40]
    out_path = OUTPUT_DIR / f'm1_{slug}.pkl'
    with open(out_path, 'wb') as f:
        pickle.dump({
            'query'      : query,
            'query_info' : query_info,
            'sentences'  : sentences,
        }, f)
    print(f'[DEBUG] Saved {len(sentences)} SentenceObjects → {out_path.name}')
    return out_path


# Uncomment to run:
# export_for_module2('Who invented the telephone?')
# export_for_module2('What did Marie Curie and Albert Einstein both contribute to physics?')
# export_for_module2('What countries are not members of NATO?')